# 04 - Baseline B3: arbol y bosque aleatorio

Se eligio **una configuracion comun de bosque** (gini, profundidad 12, hoja 20, 25 arboles, `max_features=0.5`) mediante el macro-AUC simple de `val`, con el mismo peso para A, B y C. Cada ajuste usa todo el `train` propio de su escenario. La busqueda ya esta cerrada y no se repite aqui.

Cada celda que entrena revisa primero SUS archivos en Kaggle: si ya estan, los baja a `/content/resultados/b3/...` y no entrena; si faltan, entrena solo lo que falta y lo publica.

In [ ]:
# PREPARACION DEL CUADERNO
# PROPOSITO: dejar este servidor de Colab listo SIN depender de otro cuaderno.
# - Clona el repo (o lo actualiza) e instala las librerias fijadas en requirements.txt.
# - Muestra nucleos de CPU y RAM. Para entrenar se espera CPU + RAM alta (8 nucleos, ~51 GiB).
# - Configura Kaggle: token de los secretos de Colab, de secrets.txt o pedido a mano (oculto).
# - Baja a /content/datos solo las tablas que falten en este servidor.
# Los RESULTADOS (CSV, JSON, .joblib) NO se guardan en Drive ni en el repo: viven en el dataset
# privado de Kaggle. Cada celda que entrena revisa SUS archivos en Kaggle: si ya estan, los baja
# a /content/resultados y no entrena; si faltan, entrena y los publica.
# Despues del git pull se "olvidan" los modulos ptcg_wp ya cargados para usar el codigo nuevo.
# ESPERADO: la linea de nucleos/RAM, "ya existe" o "bajado" para las 21 tablas y
# "Tablas listas | caracteristicas: 62".
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "04MainAIModelTrainingAndDevelopment"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

# Carpetas de trabajo: tablas en /content/datos y resultados en /content/resultados
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))

# Las 62 caracteristicas del modelo principal (sin ids de carta ni columnas meta)
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)
SEED = 2026
SCENARIOS = ("A", "B", "C")

# Trazabilidad que se guarda en cada registro.json nuevo
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

## Busqueda (historica, solo lectura)

Optuna con `GroupKFold` de 5 pliegues por `game_id` dentro de `train`, para arbol y bosque. `b3_cv_X.csv` tiene cada prueba y pliegue; de ahi sale la finalista por log loss que se reajusta abajo.

In [ ]:
# BUSQUEDA B3 (SOLO LECTURA)
# PROPOSITO: bajar de Kaggle los CSV de la busqueda y mostrar la finalista de cada escenario.
# No entrena.
# ESPERADO: en los tres escenarios la finalista por log loss es el bosque gini, profundidad 12,
# hoja 20, 25 arboles, max_features 0.5 (trial 1).
from ptcg_wp.finalist_evidence import b3_selected_candidate

RUTAS_BUSQUEDA_B3 = ([f"b3/busqueda/b3_cv_{e}.csv" for e in SCENARIOS]
                     + ["b3/busqueda/b3_candidatas_B.csv", "b3/busqueda/b3_candidatas_C.csv"])
if store.listo(RUTAS_BUSQUEDA_B3):
    finalistas = pd.DataFrame([b3_selected_candidate(RESULTADOS / f"b3/busqueda/b3_cv_{e}.csv", e)
                               for e in SCENARIOS])
    display(finalistas.round(4))
else:
    print("La busqueda de B3 no esta en Kaggle.")

## Modelo final de B3 por escenario

Para cada escenario: si `b3_val_X.csv` y el `.joblib` del bosque ya estan en Kaggle, se bajan y se verifican (hiperparametros, `random_state` 2030, scikit-learn 1.7.2 y uso de todo `train`). Si falta alguno, se reajusta solo el bosque finalista con todo `train` del escenario, se mide en `val` y se publica. Las metricas de CV que exige el ajuste se recuperan de los cinco pliegues registrados.

In [ ]:
# B3 FINAL: BAJAR DE KAGGLE O REAJUSTAR EL BOSQUE FINALISTA, Y VERIFICAR
# PROPOSITO: dejar en /content/resultados/b3/final/escenario_X/ el bosque y sus metricas de val.
# - Si ambos archivos estan en Kaggle: se bajan y NO se entrena.
# - Si falta alguno: se reajusta solo ese escenario (semilla efectiva 2030, como el original).
# ESPERADO: AUC val del bosque elegido 0.8029 (A), 0.7609 (B), 0.8063 (C).
import joblib
from ptcg_wp.b3 import fit_b3_candidates_val
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.finalist_evidence import b3_selected_candidate, selected_train_count, verify_full_train

PARAMETROS_B3 = {"criterion": "gini", "max_depth": 12, "min_samples_leaf": 20,
                 "n_estimators": 25, "max_features": 0.5}
for escenario in SCENARIOS:
    base = f"b3/final/escenario_{escenario}"
    csv_val = f"{base}/b3_val_{escenario}.csv"
    modelo = f"{base}/b3_{escenario}_B3_bosque_log_loss.joblib"
    print(f"--- Escenario {escenario} ---")

    def entrenar_b3(escenario=escenario, base=base):
        cv = f"b3/busqueda/b3_cv_{escenario}.csv"
        assert store.listo([cv]), "Falta la busqueda de B3 en Kaggle"
        candidata = pd.DataFrame([b3_selected_candidate(RESULTADOS / cv, escenario)])
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES))
        inicio = time.perf_counter()
        fit_b3_candidates_val(stream, FEATURES, candidata, max_train_games=None, seed=2030,
                              n_jobs=2, max_ram_gib=10.0, cache_dir="/content",
                              output_dir=RESULTADOS / base, verbose=True)
        print(f"B3 {escenario}: reajuste en {time.perf_counter() - inicio:.0f} s")

    store.asegurar([csv_val, modelo], entrenar_b3, f"B3 final escenario {escenario}")

    # Verificacion de metadatos del modelo contra su CSV de val
    item = joblib.load(RESULTADOS / modelo)
    assert item.get("escenario") == escenario and list(item["features"]) == list(FEATURES)
    assert item.get("seleccionada_por") == "log_loss" and int(item.get("trial")) == 1
    esperado = selected_train_count(RESULTADOS / csv_val, escenario, "B3")
    verify_full_train(item, escenario, "B3", esperado)
    parametros = item["model"].get_params()
    assert all(parametros[k] == v for k, v in PARAMETROS_B3.items()), parametros
    assert parametros["bootstrap"] is False and parametros["random_state"] == 2030
    assert item.get("version_sklearn") == "1.7.2"
    print(f"B3 {escenario} | train completo: {esperado} partidas | SHA256: "
          f"{store.sha256(RESULTADOS / modelo)[:16]}...")
    val = pd.read_csv(RESULTADOS / csv_val)
    elegida = val["modelo"].eq("B3_bosque") & val["seleccionada_por"].eq("log_loss")
    display(val.loc[elegida & val["etapa"].eq("global")].round(4))